In [28]:
import pandas as pd
import yfinance as yfin
import matplotlib.pyplot as plt
import datetime as dt
import numpy as np
import scipy as scp
from scipy import stats

In [29]:
Ticker = ['^NSEI']

# Indicate the start and end dates
start = dt.datetime.now().replace(year=dt.datetime.now().year - 1)
end = dt.datetime.now()

nifty = yfin.download(Ticker, start = start, end = end)
nifty

[*********************100%***********************]  1 of 1 completed


Price,Close,High,Low,Open,Volume
Ticker,^NSEI,^NSEI,^NSEI,^NSEI,^NSEI
Date,,,,,
2025-05-19,24945.449219,25062.949219,24916.650391,25005.349609,255300
2025-05-20,24683.900391,25010.349609,24669.699219,24996.199219,414800
2025-05-21,24813.449219,24946.199219,24685.349609,24744.250000,332700
2025-05-22,24609.699219,24737.500000,24462.400391,24733.949219,403300
2025-05-23,24853.150391,24909.050781,24614.050781,24639.500000,270500
...,...,...,...,...,...
2026-05-13,23412.599609,23582.949219,23262.550781,23362.449219,415400
2026-05-14,23689.599609,23777.199219,23426.550781,23530.250000,428700


In [30]:
nifty['todayReturns'] = nifty['Close'].pct_change(1) * 100
nifty['tomorrowReturns'] = nifty.todayReturns.shift(-1)

nifty

Price,Close,High,Low,Open,Volume,todayReturns,tomorrowReturns
Ticker,^NSEI,^NSEI,^NSEI,^NSEI,^NSEI,,
Date,,,,,,,
2025-05-19,24945.449219,25062.949219,24916.650391,25005.349609,255300,NaN,-1.048483
2025-05-20,24683.900391,25010.349609,24669.699219,24996.199219,414800,-1.048483,0.524831
2025-05-21,24813.449219,24946.199219,24685.349609,24744.250000,332700,0.524831,-0.821127
2025-05-22,24609.699219,24737.500000,24462.400391,24733.949219,403300,-0.821127,0.989249
2025-05-23,24853.150391,24909.050781,24614.050781,24639.500000,270500,0.989249,0.595498
...,...,...,...,...,...,...,...
2026-05-13,23412.599609,23582.949219,23262.550781,23362.449219,415400,0.141358,1.183124
2026-05-14,23689.599609,23777.199219,23426.550781,23530.250000,428700,1.183124,-0.194599


In [32]:
returns = nifty[['todayReturns', 'tomorrowReturns']].dropna()

returns

Price,todayReturns,tomorrowReturns
Ticker,,
Date,,
2025-05-20,-1.048483,0.524831
2025-05-21,0.524831,-0.821127
2025-05-22,-0.821127,0.989249
2025-05-23,0.989249,0.595498
2025-05-26,0.595498,-0.699772
...,...,...
2026-05-12,-1.831968,0.141358
2026-05-13,0.141358,1.183124


In [40]:
X_no_intercept = returns[['todayReturns']].values
X = np.column_stack([np.ones(len(X_no_intercept)), X_no_intercept]) 
Y = returns[['tomorrowReturns']].values

beta = np.linalg.inv(X.T @ X) @ X.T @ Y

In [41]:
slope, intercept, r_value, p_value, std_err = stats.linregress(returns['todayReturns'], returns['tomorrowReturns'])

In [42]:
print(f"linregress : {slope}")
print(f"numpy : {beta}")

linregress : -0.059690182886968916
numpy : [[-0.01579197]
 [-0.05969018]]


In [43]:
print(f"Slope:     {slope:.6f}")
print(f"Intercept: {intercept:.6f}")
print(f"R²:        {r_value**2:.6f}")
print(f"P-value:   {p_value:.6f}")

Slope:     -0.059690
Intercept: -0.015792
R²:        0.003586
P-value:   0.349616


In [57]:
# Test if Nifty mean daily return significantly different from 0
# Null hypothesis: mean = 0
# Alternate Hypthesis: mean != 0
# level of significance = 0.05
dailyReturns = nifty['todayReturns'].dropna()
dailyReturns

Date
2025-05-20   -1.048483
2025-05-21    0.524831
2025-05-22   -0.821127
2025-05-23    0.989249
2025-05-26    0.595498
                ...   
2026-05-13    0.141358
2026-05-14    1.183124
2026-05-15   -0.194599
2026-05-18    0.027277
2026-05-19   -0.135092
Name: todayReturns, Length: 247, dtype: float64

In [59]:
print(dailyReturns.mean())
print(dailyReturns.std())

-0.018878679126934753
0.8085323152767319


In [61]:
testResults = stats.ttest_1samp(dailyReturns, popmean = 0)
print(testResults)

Ttest_1sampResult(statistic=-0.3669633562831837, pvalue=0.7139618125462188)


In [ ]:
#since p-value is > 0.05, null hypothesis cannot be rejected
# mean return is 0